# 🚧 Pothole Detection — YOLOv11 Training (Google Colab)

**This is the 100% automated Colab version.**
It will automatically download the dataset, train the model, and save `best.pt` directly to your Google Drive so you don't lose it if you close the tab.

### Steps:
1. Go to **Runtime -> Change runtime type** and select **T4 GPU**.
2. Click **Runtime -> Run all**.
3. It will ask for permission to mount your Google Drive. Click **Allow**.
4. Wait ~2 hours. Your model will appear in your Google Drive!

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────
# This ensures your trained model is saved permanently to your Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RoadSafe_Model'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Google Drive mounted! Model will save to: {DRIVE_DIR}')

In [ ]:
# ── Step 2: Install Dependencies & Prevent Crashes ────────────────────────
!pip install ultralytics "numpy<2.0.0" "opencv-python<4.10.0" "opencv-python-headless<4.10.0" -q
print('✅ Dependencies installed!')

In [ ]:
# ── Step 3: GPU Check ─────────────────────────────────────────────────────
import torch
print('=' * 50)
print(f'  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
else:
    print('  ⚠️  No GPU — Go to Runtime -> Change runtime type -> T4 GPU')
print('=' * 50)

In [ ]:
# ── Step 4: Automatically Download Dataset ────────────────────────────────
!wget -q -O pothole_dataset.zip https://github.com/jaygala24/pothole-detection/releases/download/v1.0.0/Pothole.Dataset.IVCNZ.zip
!unzip -q pothole_dataset.zip -d /content/downloaded_dataset
print('✅ Dataset downloaded and extracted successfully!')

In [ ]:
# ── Step 5: Organise Dataset ──────────────────────────────────────────────
import os, shutil, random
from pathlib import Path

INPUT_DIR = Path('/content/downloaded_dataset')
MERGED_DIR = Path('/content/merged_dataset')

for split in ['train', 'val', 'test']:
    (MERGED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

print("Finding label files...")
labels_map = {}
for p in INPUT_DIR.rglob('*.txt'):
    if p.name.lower() not in ['readme.txt', 'classes.txt', 'requirements.txt']:
        labels_map[p.stem] = p

EXTS = {'.jpg', '.jpeg', '.png'}
pairs = []
for img_p in INPUT_DIR.rglob('*'):
    if img_p.suffix.lower() in EXTS:
        if img_p.stem in labels_map:
            pairs.append((img_p, labels_map[img_p.stem]))

if len(pairs) == 0:
    raise ValueError("❌ Dataset extraction failed.")

random.shuffle(pairs)
n = len(pairs)
cuts = [int(n * 0.75), int(n * 0.90)]
splits = {
    'train': pairs[:cuts[0]],
    'val':   pairs[cuts[0]:cuts[1]],
    'test':  pairs[cuts[1]:]
}

def remap_label_to_0(label_path, out_path):
    lines = []
    try:
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'
                    lines.append(' '.join(parts))
    except Exception: return False
    if lines:
        with open(out_path, 'w') as f:
            f.write('\n'.join(lines))
        return True
    return False

added = 0
for split, sp in splits.items():
    for i, (ip, lp) in enumerate(sp):
        name = f'pothole_{i:05d}'
        dst_img = MERGED_DIR / split / 'images' / (name + ip.suffix)
        dst_lbl = MERGED_DIR / split / 'labels' / (name + '.txt')
        shutil.copy2(ip, dst_img)
        if remap_label_to_0(lp, dst_lbl):
            added += 1

print(f"✅ Organised {added} pairs.")

In [ ]:
# ── Step 6: Write dataset.yaml ────────────────────────────────────────────
import yaml

yaml_path = MERGED_DIR / 'dataset.yaml'
cfg = {
    'path':  str(MERGED_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    1,
    'names': ['pothole'],
}
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ dataset.yaml created!')

In [ ]:
# ── Step 7: TRAIN YOLOv11 & Save to Google Drive ──────────────────────────
# ⏱️ This takes a couple of hours. Do NOT close the tab!

from ultralytics import YOLO
import shutil

model = YOLO('yolo11m.pt')  # YOLOv11 Medium

results = model.train(
    data         = str(yaml_path),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    patience     = 20,
    save         = True,
    project      = '/content/runs',
    name         = 'pothole_yolo11',
    device       = 0,
)

print('\n✅ Training complete!')

# Copy the final model to your Google Drive so you don't lose it!
best_pt_path = f'{results.save_dir}/weights/best.pt'
drive_save_path = f'{DRIVE_DIR}/best.pt'
shutil.copy2(best_pt_path, drive_save_path)

print(f'🎉 best.pt successfully saved to your Google Drive at: {drive_save_path}')